In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [2]:
from pathlib import Path

DIR = Path(r"/kaggle/input/competitions/house-prices-advanced-regression-techniques")

train_path = DIR / "train.csv"
test_path  = DIR / "test.csv"
subm_path  = DIR / "sample_submission.csv"

train = pd.read_csv(train_path)
test  = pd.read_csv(test_path)

TARGET  = "SalePrice"
test_id = test.Id

train.drop('Id', axis=1, inplace=True)
test.drop('Id', axis=1, inplace=True)

assert train.columns.tolist() == test.columns.tolist() + [TARGET]

N_SPLITS = 5

In [190]:
# Preprocessing
train.fillna(0, inplace=True)
test.fillna(0, inplace=True)

num_cols = train.select_dtypes(exclude=['object', 'string']).drop(TARGET, axis=1)

display(num_cols.columns.tolist())

X_train = num_cols.to_numpy()
y_train = train[TARGET].to_numpy()
X_test  = test[num_cols.columns.tolist()].to_numpy()

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
# X_train = scaler.fit_transform(X_train)
# X_test  = scaler.transform(X_test)

['MSSubClass',
 'LotFrontage',
 'LotArea',
 'OverallQual',
 'OverallCond',
 'YearBuilt',
 'YearRemodAdd',
 'MasVnrArea',
 'BsmtFinSF1',
 'BsmtFinSF2',
 'BsmtUnfSF',
 'TotalBsmtSF',
 '1stFlrSF',
 '2ndFlrSF',
 'LowQualFinSF',
 'GrLivArea',
 'BsmtFullBath',
 'BsmtHalfBath',
 'FullBath',
 'HalfBath',
 'BedroomAbvGr',
 'KitchenAbvGr',
 'TotRmsAbvGrd',
 'Fireplaces',
 'GarageYrBlt',
 'GarageCars',
 'GarageArea',
 'WoodDeckSF',
 'OpenPorchSF',
 'EnclosedPorch',
 '3SsnPorch',
 'ScreenPorch',
 'PoolArea',
 'MiscVal',
 'MoSold',
 'YrSold']

In [191]:
# Metrics
def MSE(y_pred, y):
    return ((y_pred - y)**2).mean()

In [17]:
X_train.shape

(1460, 36)

In [101]:
# testing
W = np.random.rand(X_train.shape[1], 1)
B = np.random.random()
lr = 1e-5
last = None

def fit(X, y):
    global W, B, lr, last
    
    y_pred = predict(X)

    if last:
        if MSE(y_pred, y) < last:
            print(f"Improved: {MSE(y_pred, y) - last}")
        
        else:
            print(f"Downgraded: {MSE(y_pred, y) - last}")
            
        last = MSE(y_pred, y)

    else:
        last = MSE(y_pred, y)
        print(last)
    
    L = (y_pred - y)
    a_j = X

    m = X.shape[0]
    W += lr * (a_j.T @ L) / m
    B += lr * np.sum(L) / m

def predict(X):
    global W, B

    return X @ W + B

In [105]:
# display(X_train[0])

x = X_train[0].reshape(1, -1)
y = y_train[0].reshape(-1, 1)

fit(x, y)

Downgraded: 2.7331165952353776e+28
(1, 1) (1, 36) (36, 1)


In [193]:
# Model building
class LinearRegression_():
    def __init__(self):
        self.w  = None
        self.b  = np.random.random()
        self.lr = 0.03
        self.epoch = 6000
        self.best = float('inf')
    
    def fit(self, X, y):
        y = y.reshape(-1, 1)
        assert len(X.shape) == 2 and len(y.shape) == 2
        
        # Initialization
        self.w = np.random.rand(X.shape[1], 1)
        is_imp = False

        for epoch in range(self.epoch):
            # Single update interation
            y_pred = self.predict(X)
            L   = (y_pred - y)
            a_j = X
            m = X.shape[0]

            self.w = self.w - (self.lr / m * (a_j.T @ L))
            self.b = self.b - (self.lr / m * float(np.sum(L)))

            new = MSE(y_pred, y)
            if not is_imp and self.best > new:
                self.best = new
                is_imp = True
                
            if (epoch % 500 == 0):
                print("{:^10}".format(f"Epoch={epoch}"))
                print(f"MSE loss={MSE(y_pred, y)}")
                if is_imp:
                    print("Improved\n")
                else:
                    print("Not Improved\n")
                is_imp=False
    
    def predict(self, X):
        assert (self.w is not None)
        return X @ self.w + self.b

In [198]:
from sklearn.linear_model import LinearRegression

In [199]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=N_SPLITS)

for KFold, (train_idx, val_idx) in enumerate(kf.split(X_train, y_train)):
    print("{:-^40}".format(f"Fold: {KFold + 1}"))

    X_tr, y_tr = X_train[train_idx], y_train[train_idx]
    X_vl, y_vl = X_train[val_idx], y_train[val_idx]

    X_tr = scaler.fit_transform(X_tr)
    X_vl = scaler.transform(X_vl)
    
    model = LinearRegression()
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_vl)

    model = LinearRegression_()
    model.fit(X_tr, y_tr)
    y_pred_ = model.predict(X_vl)    
    
    print(f"MSE error sklearn: {MSE(y_pred, y_vl)}")
    print(f"MSE error mine: {MSE(y_pred_, y_vl)}")
    print(f"Diff: {MSE(y_pred_, y_vl) - MSE(y_pred, y_vl)}")

----------------Fold: 1-----------------
Epoch=  0   | Loss=39158787184.8724
Epoch=1000  | Loss=1262268831.0159
Epoch=2000  | Loss=1259943997.6946
Epoch=3000  | Loss=1259703613.6312
Epoch=4000  | Loss=1259670145.6964
Epoch=5000  | Loss=1259665330.5041
MSE error sklearn: 797608251.0074104
MSE error mine: 10421642790.485323
Diff: 9624034539.477913
----------------Fold: 2-----------------
Epoch=  0   | Loss=38786392496.3065
Epoch=1000  | Loss=1160882531.0808
Epoch=2000  | Loss=1157695991.3190
Epoch=3000  | Loss=1157350557.1820
Epoch=4000  | Loss=1157305966.9669
Epoch=5000  | Loss=1157300048.0076
MSE error sklearn: 1199861380.2744427
MSE error mine: 12456262153.621124
Diff: 11256400773.346682
----------------Fold: 3-----------------
Epoch=  0   | Loss=38387515940.6918
Epoch=1000  | Loss=1142500907.5667
Epoch=2000  | Loss=1140299785.8889
Epoch=3000  | Loss=1140072580.6688
Epoch=4000  | Loss=1140041213.1253
Epoch=5000  | Loss=1140036686.7975
MSE error sklearn: 1318652393.6117673
MSE error mi